In [1]:
import os
import time
import math
import json
import requests
import pandas as pd
import torch
import braintrust

from transformers import AutoTokenizer, AutoModelForSequenceClassification

/Users/izzyhurley/miniconda3/envs/community_monitor/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
OWNER = "pandas-dev"
REPO = "pandas"


GITHUB_TOKEN = os.environ["GITHUB_TOKEN"]
HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "X-GitHub-Api-Version": "2022-11-28",
}

BASE = f"https://api.github.com/repos/{OWNER}/{REPO}"


def gh_get(url, params=None):
    r = requests.get(url, headers=HEADERS, params=params, timeout=60)
    r.raise_for_status()
    return r


def gh_paginate(url, params=None, sleep=0.1):
    items = []
    while url:
        r = gh_get(url, params=params)
        data = r.json()
        if isinstance(data, list):
            items.extend(data)
        else:
            raise ValueError(f"Expected list response for {url}")
        url = r.links.get("next", {}).get("url")
        params = None
        time.sleep(sleep)
    return items


def list_issues_and_prs(state="all", since=None, per_page=100, max_items=None):
    params = {"state": state, "per_page": per_page}
    if since:
        params["since"] = since
    items = gh_paginate(f"{BASE}/issues", params=params)
    if max_items:
        items = items[:max_items]
    return items


def list_issue_comments(number):
    return gh_paginate(f"{BASE}/issues/{number}/comments", params={"per_page": 100})


def list_pr_reviews(number):
    return gh_paginate(f"{BASE}/pulls/{number}/reviews", params={"per_page": 100})


def list_pr_review_comments(number):
    return gh_paginate(f"{BASE}/pulls/{number}/comments", params={"per_page": 100})


def normalize_issue_comment(c, discussion_type, number):
    return {
        "repo": f"{OWNER}/{REPO}",
        "discussion_type": discussion_type,
        "discussion_number": number,
        "comment_source": "issue_comment",
        "comment_id": c["id"],
        "comment_created_at": c["created_at"],
        "author_login": c["user"]["login"] if c.get("user") else None,
        "author_type": c["user"]["type"] if c.get("user") else None,
        "author_association": c.get("author_association"),
        "body": c.get("body") or "",
    }


def normalize_pr_review(review, number):
    return {
        "repo": f"{OWNER}/{REPO}",
        "discussion_type": "pull_request",
        "discussion_number": number,
        "comment_source": "review",
        "comment_id": review["id"],
        "comment_created_at": review["submitted_at"] or review["created_at"],
        "author_login": review["user"]["login"] if review.get("user") else None,
        "author_type": review["user"]["type"] if review.get("user") else None,
        "author_association": review.get("author_association"),
        "body": review.get("body") or "",
    }


def normalize_pr_review_comment(c, number):
    return {
        "repo": f"{OWNER}/{REPO}",
        "discussion_type": "pull_request",
        "discussion_number": number,
        "comment_source": "review_comment",
        "comment_id": c["id"],
        "comment_created_at": c["created_at"],
        "author_login": c["user"]["login"] if c.get("user") else None,
        "author_type": c["user"]["type"] if c.get("user") else None,
        "author_association": c.get("author_association"),
        "body": c.get("body") or "",
    }


def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    return tokenizer, model, device


def score_texts(texts, tokenizer, model, device, batch_size=32, max_length=256):
    probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits

            if logits.ndim == 2 and logits.shape[1] == 1:
                batch_probs = torch.sigmoid(logits[:, 0])
            elif logits.ndim == 2 and logits.shape[1] == 2:
                batch_probs = torch.softmax(logits, dim=1)[:, 1]
            else:
                raise ValueError(f"Unexpected logits shape: {tuple(logits.shape)}")

        probs.extend(batch_probs.detach().cpu().tolist())
    return probs


def aggregate_thread(df_thread):
    # Called with include_groups=False (pandas 3.0) — group key columns are
    # excluded from df_thread and come back via reset_index() at the call site.
    probs = df_thread["toxicity_prob"].tolist()
    comment_count = len(df_thread)
    unique_commenters = df_thread["author_login"].nunique(dropna=True)
    max_prob = max(probs) if probs else None
    mean_prob = sum(probs) / len(probs) if probs else None

    ge_04 = int((df_thread["toxicity_prob"] >= 0.4).sum())
    ge_07 = int((df_thread["toxicity_prob"] >= 0.7).sum())

    first_toxic_idx = None
    flagged = df_thread[df_thread["toxicity_prob"] >= 0.7].sort_values("comment_created_at")
    if not flagged.empty:
        first_comment_id = flagged.iloc[0]["comment_id"]
        ordered = df_thread.sort_values("comment_created_at").reset_index(drop=True)
        first_toxic_idx = int(ordered.index[ordered["comment_id"] == first_comment_id][0]) + 1

    is_newcomer_involved = bool(
        (df_thread["author_association"] == "FIRST_TIME_CONTRIBUTOR").any()
    )

    return pd.Series({
        "comment_count": comment_count,
        "unique_commenter_count": unique_commenters,
        "max_toxicity_prob": max_prob,
        "mean_toxicity_prob": mean_prob,
        "comments_ge_0_4": ge_04,
        "comments_ge_0_7": ge_07,
        "first_toxic_comment_index": first_toxic_idx,
        "is_newcomer_involved": is_newcomer_involved,
    })


def suggest_stratum(row):
    if row["comment_count"] <= 5 and (row["max_toxicity_prob"] or 0) < 0.2:
        return "control_candidate"
    if row["comments_ge_0_7"] >= 1:
        return "clearly_toxic_candidate"
    if row["comments_ge_0_4"] >= 1 and (row["max_toxicity_prob"] or 0) < 0.7:
        return "borderline_candidate"
    if row["comment_count"] >= 15 and (row["max_toxicity_prob"] or 0) < 0.4:
        return "heated_not_toxic_candidate"
    return "other"

In [3]:
%pip install protobuf


Note: you may need to restart the kernel to use updated packages.


In [32]:
# ── Fetch issues/PRs and all comment types ────────────────────────────────────
SINCE = "2023-01-01T00:00:00Z"
# MAX_ITEMS = 10  # reduced for a quick test run
MAX_ITEMS = 2000

print("Fetching issues and PRs...")
items = list_issues_and_prs(state="all", since=SINCE, max_items=MAX_ITEMS)
print(f"  → {len(items)} items fetched")


Fetching issues and PRs...
  → 2000 items fetched


In [33]:
from pathlib import Path
import pickle
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
cache_file = CACHE_DIR / f"issues_prs_{SINCE[:10]}_{MAX_ITEMS}.pkl"

# if cache_file.exists():
#     print(f"Loading cached items from {cache_file}...")
#     with open(cache_file, "rb") as f:
#         items = pickle.load(f)
#     print(f"  → {len(items)} items loaded from cache")
# else:
# print("Fetching issues and PRs...")
items = list_issues_and_prs(state="all", since=SINCE, max_items=MAX_ITEMS)
print(f"  → {len(items)} items fetched")

with open(cache_file, "wb") as f:
    pickle.dump(items, f)

print(f"  → saved to {cache_file}")

Fetching issues and PRs...
  → 2000 items fetched
  → saved to cache/issues_prs_2023-01-01_2000.pkl


In [34]:
rows = []
for i, item in enumerate(items):
    number = item["number"]
    is_pr = "pull_request" in item
    dtype = "pull_request" if is_pr else "issue"

    for c in list_issue_comments(number):
        rows.append(normalize_issue_comment(c, dtype, number))

    if is_pr:
        for rev in list_pr_reviews(number):
            if rev.get("body", "").strip():
                rows.append(normalize_pr_review(rev, number))
        for rc in list_pr_review_comments(number):
            rows.append(normalize_pr_review_comment(rc, number))

    if (i + 1) % 50 == 0:
        print(f"  → {i + 1}/{len(items)} items processed")

df_comments = pd.DataFrame(rows)
print(f"\nTotal comments collected: {len(df_comments)}")

  → 50/2000 items processed
  → 100/2000 items processed
  → 150/2000 items processed
  → 200/2000 items processed
  → 250/2000 items processed
  → 300/2000 items processed
  → 350/2000 items processed
  → 400/2000 items processed
  → 450/2000 items processed
  → 500/2000 items processed
  → 550/2000 items processed
  → 600/2000 items processed
  → 650/2000 items processed
  → 700/2000 items processed
  → 750/2000 items processed
  → 800/2000 items processed
  → 850/2000 items processed
  → 900/2000 items processed
  → 950/2000 items processed
  → 1000/2000 items processed
  → 1050/2000 items processed
  → 1100/2000 items processed
  → 1150/2000 items processed
  → 1200/2000 items processed
  → 1250/2000 items processed
  → 1300/2000 items processed
  → 1350/2000 items processed
  → 1400/2000 items processed
  → 1450/2000 items processed
  → 1500/2000 items processed
  → 1550/2000 items processed
  → 1600/2000 items processed
  → 1650/2000 items processed
  → 1700/2000 items processed


In [35]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


# Define device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "toxishield/toxic-classifier-38k"
BASE_TOKENIZER = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(BASE_TOKENIZER)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)  # Move model to device

# Score comments for toxicity
print("Scoring comments for toxicity...")
df_comments["toxicity_prob"] = score_texts(
    df_comments["body"].fillna("").tolist(), tokenizer, model, device
)
print("  → done")
df_comments.head()

Scoring comments for toxicity...
  → done


,repo,discussion_type,discussion_number,comment_source,comment_id,comment_created_at,author_login,author_type,author_association,body,toxicity_prob
0,pandas-dev/pandas,issue,65373,issue_comment,4322369021,2026-04-26T15:31:20Z,Abineshabee,User,CONTRIBUTOR,"Hi @Alvaro-Kothe, I’d like to work on this iss...",0.000561
1,pandas-dev/pandas,pull_request,65372,review_comment,3143677870,2026-04-26T15:07:05Z,jbrockmendel,User,MEMBER,i suspect this is slightly slower than\n\n```\...,0.000514
2,pandas-dev/pandas,pull_request,65372,review_comment,3143679052,2026-04-26T15:07:58Z,jbrockmendel,User,MEMBER,why not just remove these from the parametriza...,0.000619
3,pandas-dev/pandas,pull_request,65371,review,4176965329,2026-04-26T13:33:24Z,rhshadrach,User,MEMBER,Thanks for the PR! See my suggestion in the li...,0.000468
4,pandas-dev/pandas,pull_request,65371,review_comment,3143537493,2026-04-26T13:21:43Z,rhshadrach,User,MEMBER,This argument has possible effects for all inp...,0.000515


In [36]:
df_comments.shape

(6266, 11)

In [37]:
# ── Aggregate to thread level + assign strata ─────────────────────────────────
df_threads = (
    df_comments
    .groupby(["repo", "discussion_type", "discussion_number"])
    .apply(aggregate_thread, include_groups=False)
    .reset_index()
)
df_threads["stratum"] = df_threads.apply(suggest_stratum, axis=1)

print(df_threads["stratum"].value_counts().to_string())
df_threads.head(10)

stratum
control_candidate             1306
other                          238
heated_not_toxic_candidate      53
clearly_toxic_candidate         25
borderline_candidate             2


,repo,discussion_type,discussion_number,comment_count,unique_commenter_count,max_toxicity_prob,mean_toxicity_prob,comments_ge_0_4,comments_ge_0_7,first_toxic_comment_index,is_newcomer_involved,stratum
0,pandas-dev/pandas,issue,63371,10,4,0.000974,0.000571,0,0,NaN,False,other
1,pandas-dev/pandas,issue,63372,1,1,0.000565,0.000565,0,0,NaN,False,control_candidate
2,pandas-dev/pandas,issue,63373,4,3,0.000976,0.000641,0,0,NaN,False,control_candidate
3,pandas-dev/pandas,issue,63374,2,2,0.000682,0.000649,0,0,NaN,False,control_candidate
4,pandas-dev/pandas,issue,63375,1,1,0.000472,0.000472,0,0,NaN,False,control_candidate
5,pandas-dev/pandas,issue,63376,2,1,0.000648,0.000563,0,0,NaN,False,control_candidate
6,pandas-dev/pandas,issue,63377,2,1,0.000754,0.000656,0,0,NaN,False,control_candidate
7,pandas-dev/pandas,issue,63378,4,3,0.000648,0.000541,0,0,NaN,False,control_candidate
8,pandas-dev/pandas,issue,63385,3,2,0.000648,0.000579,0,0,NaN,False,control_candidate
9,pandas-dev/pandas,issue,63388,3,2,0.002245,0.001618,0,0,NaN,False,control_candidate


In [40]:
# ── Stratified sample: 5 per stratum ─────────────────────────────────────────
SAMPLES_PER_STRATUM = 20
TARGET_STRATA = [
    "clearly_toxic_candidate",
    "borderline_candidate",
    "heated_not_toxic_candidate",
    "control_candidate",
]

sampled_parts = []
for stratum in TARGET_STRATA:
    group = df_threads[df_threads["stratum"] == stratum]
    n = min(SAMPLES_PER_STRATUM, len(group))
    if n < SAMPLES_PER_STRATUM:
        print(f"  CAUTION! only {n} rows for stratum '{stratum}' (wanted {SAMPLES_PER_STRATUM})")
    sampled_parts.append(group.sample(n, random_state=42))

sampled = pd.concat(sampled_parts).reset_index(drop=True)
print(f"\nSampled {len(sampled)} threads:")
print(sampled["stratum"].value_counts().to_string())
sampled

  CAUTION! only 2 rows for stratum 'borderline_candidate' (wanted 20)

Sampled 62 threads:
stratum
clearly_toxic_candidate       20
heated_not_toxic_candidate    20
control_candidate             20
borderline_candidate           2


,repo,discussion_type,discussion_number,comment_count,unique_commenter_count,max_toxicity_prob,mean_toxicity_prob,comments_ge_0_4,comments_ge_0_7,first_toxic_comment_index,is_newcomer_involved,stratum
0,pandas-dev/pandas,pull_request,63613,17,3,0.995017,0.059174,1,1,7.0,False,clearly_toxic_candidate
1,pandas-dev/pandas,pull_request,64422,13,2,0.999393,0.077511,1,1,1.0,False,clearly_toxic_candidate
2,pandas-dev/pandas,issue,63444,18,4,0.997664,0.155225,3,3,3.0,False,clearly_toxic_candidate
3,pandas-dev/pandas,pull_request,65127,26,3,0.997785,0.039166,1,1,15.0,False,clearly_toxic_candidate
4,pandas-dev/pandas,pull_request,63915,90,3,0.999563,0.011926,1,1,23.0,False,clearly_toxic_candidate
...,...,...,...,...,...,...,...,...,...,...,...,...
57,pandas-dev/pandas,pull_request,64735,3,2,0.002087,0.001099,0,0,NaN,False,control_candidate
58,pandas-dev/pandas,pull_request,64700,1,1,0.001264,0.001264,0,0,NaN,False,control_candidate
59,pandas-dev/pandas,pull_request,64319,2,2,0.000601,0.000591,0,0,NaN,False,control_candidate
60,pandas-dev/pandas,pull_request,64289,2,1,0.000539,0.000514,0,0,NaN,False,control_candidate


In [42]:
# ── Assemble dataset rows ─────────────────────────────────────────────────────
dataset_rows = []
for _, row in sampled.iterrows():
    dtype_short = "pr" if row["discussion_type"] == "pull_request" else "issue"
    number = int(row["discussion_number"])
    row_id = f"pandas-{dtype_short}-{number}"
    url_segment = "pull" if row["discussion_type"] == "pull_request" else "issues"

    dataset_rows.append({
        "id": row_id,
        "repo": row["repo"],
        "discussion_type": row["discussion_type"],
        "discussion_number": number,
        "url": f"https://github.com/{row['repo']}/{url_segment}/{number}",
        "ground_truth": {
            "is_toxic": None,           # TODO: annotate in Braintrust
            "toxicity_labels": [],      # TODO: annotate — see schema in eval design doc
            "severity": None,           # TODO: annotate — low / medium / high
            "problematic_snippet": "",  # TODO: annotate
            "gold_response": "",        # TODO: annotate
        },
        "metadata": {
            "comment_count": int(row["comment_count"]),
            "first_toxic_comment_index": (
                    int(row["first_toxic_comment_index"])
                    if not pd.isna(row["first_toxic_comment_index"]) else None
                ),
            "is_newcomer_involved": bool(row["is_newcomer_involved"]),
            "max_toxicity_prob": (
                float(row["max_toxicity_prob"])
                if row["max_toxicity_prob"] is not None else None
            ),
            "mean_toxicity_prob": (
                float(row["mean_toxicity_prob"])
                if row["mean_toxicity_prob"] is not None else None
            ),
            "stratum": row["stratum"],
        },
    })

print(f"Assembled {len(dataset_rows)} rows")
print(json.dumps(dataset_rows[0], indent=2))

Assembled 62 rows
{
  "id": "pandas-pr-63613",
  "repo": "pandas-dev/pandas",
  "discussion_type": "pull_request",
  "discussion_number": 63613,
  "url": "https://github.com/pandas-dev/pandas/pull/63613",
  "ground_truth": {
    "is_toxic": null,
    "toxicity_labels": [],
    "severity": null,
    "problematic_snippet": "",
    "gold_response": ""
  },
  "metadata": {
    "comment_count": 17,
    "first_toxic_comment_index": 7,
    "is_newcomer_involved": false,
    "max_toxicity_prob": 0.9950174689292908,
    "mean_toxicity_prob": 0.05917410150184022,
    "stratum": "clearly_toxic_candidate"
  }
}


In [45]:
os.environ["BRAINTRUST_API_KEY"]

KeyError: 'BRAINTRUST_API_KEY'

In [47]:
# ── Save locally + upload to Braintrust ──────────────────────────────────────
output_path = "community-health-v1.json"
with open(output_path, "w") as f:
    json.dump(dataset_rows, f, indent=2)
print(f"Saved to {output_path}")

# Requires BRAINTRUST_API_KEY in env
ds = braintrust.init_dataset(
    project="community-health-eval",
    name="community-health-v1",
    description="pandas-dev/pandas discussions for community health first responder eval",
)

for row in dataset_rows:
    ds.insert(
        input={
            "repo": row["repo"],
            "discussion_type": row["discussion_type"],
            "discussion_number": row["discussion_number"],
            "url": row["url"],
        },
        expected=row["ground_truth"],
        metadata=row["metadata"],
        id=row["id"],
    )

ds.flush()
print(f"Uploaded {len(dataset_rows)} rows → Braintrust 'community-health-v1'")

Saved to community-health-v1.json
Uploaded 62 rows → Braintrust 'community-health-v1'
